# 02 — Manual Backpropagation (Part A)

Chạy `src/manual_nn.ManualMLP` (forward + backward tự viết bằng NumPy, không autograd) trên ví dụ 2 lớp đã verify ở `docs/phase2_math_content.md` mục 4.4, rồi đối chiếu với PyTorch autograd.

In [1]:
import sys; sys.path.insert(0, '..')
import numpy as np
import torch
from src.manual_nn import ManualMLP


In [2]:
# Kien truc + tham so dung DUNG voi vi du da verify trong Phase 2
net = ManualMLP(layer_dims=[2,3,1], activation='relu', scheme='he', seed=0)
net.W[0] = np.array([[0.30,-0.30,0.10],[0.20,0.10,0.15]])
net.b[0] = np.array([0.10,-0.20,0.05])
net.W[1] = np.array([[0.20],[-0.30],[0.40]])
net.b[1] = np.array([0.05])

x = np.array([[1.0, 1.0]])
y = np.array([[1.0]])
yhat = net.forward(x)
L = net.loss(yhat, y)
grads = net.backward(yhat, y)
print('yhat =', yhat, ' L =', L)
print('dW1 =\n', grads['dW'][0])
print('db1 =', grads['db'][0])
print('dW2 =\n', grads['dW'][1])
print('db2 =', grads['db'][1])

yhat = [[0.29]]  L = 0.25205
dW1 =
 [[-0.142  0.    -0.284]
 [-0.142  0.    -0.284]]
db1 = [-0.142  0.    -0.284]
dW2 =
 [[-0.426]
 [ 0.   ]
 [-0.213]]
db2 = [-0.71]


So khớp với Bảng số liệu Mục 4.4: $\hat y=0.29$, $L=0.25205$, $\partial L/\partial W_1$ hai hàng bằng nhau $=(-0.142,\,0,\,-0.284)$ — khớp chính xác.

## Đối chiếu PyTorch autograd

In [3]:
import torch.nn as nn
torch.set_default_dtype(torch.float64)
lin1 = nn.Linear(2,3); lin2 = nn.Linear(3,1)
with torch.no_grad():
    lin1.weight.copy_(torch.tensor(net.W[0].T)); lin1.bias.copy_(torch.tensor(net.b[0]))
    lin2.weight.copy_(torch.tensor(net.W[1].T)); lin2.bias.copy_(torch.tensor(net.b[1]))
xt = torch.tensor(x); yt = torch.tensor(y)
pred = lin2(torch.relu(lin1(xt)))
loss = 0.5*((pred-yt)**2).sum()/x.shape[0]
loss.backward()
print('autograd dW1 =\n', lin1.weight.grad.numpy())
print('manual   dW1 =\n', grads['dW'][0].T)
print('max abs diff:', float((lin1.weight.grad.numpy()-grads['dW'][0].T).__abs__().max()))

autograd dW1 =
 [[-0.142 -0.142]
 [ 0.     0.   ]
 [-0.284 -0.284]]
manual   dW1 =
 [[-0.142 -0.142]
 [ 0.     0.   ]
 [-0.284 -0.284]]
max abs diff: 0.0
